In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import matplotlib.pyplot as plt
import numpy as np
from torch import optim
from sklearn.datasets import make_moons

from stochinter.models import PotentialNet, DirectNet
from stochinter.utils import solve_ode, get_gradient_field

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# parameters
SEED = 42
BATCH_SIZE = 1024
EPOCHS = 10_000
LEARNING_RATE = 3e-3

In [ ]:
torch.manual_seed(SEED)
batch_size = BATCH_SIZE
epochs = EPOCHS

net_potential = PotentialNet()
net_direct = DirectNet()

opt_pot = optim.Adam(net_potential.parameters(), lr=LEARNING_RATE)
opt_dir = optim.Adam(net_direct.parameters(), lr=LEARNING_RATE)

print("Training of models")

for epoch in range(epochs):
    x0 = torch.randn(batch_size, 2)
    x1_np, _ = make_moons(n_samples=batch_size, noise=0.05)
    x1 = torch.tensor((x1_np - np.array([0.5, 0.25])) * 2.0, dtype=torch.float32)
    
    z = torch.randn(batch_size, 2)
    t = torch.rand(batch_size, 1)
    
    # Interpolation
    x_t = (1 - t) * x0 + t * x1 + torch.sqrt(2*t*(1-t))*z 
    
    # Target for vector field
    omega = 3.0
    v_tornado = omega * torch.stack([-x_t[:, 1], x_t[:, 0]], dim=1)
    v_target = (x1 - x0 + z*(1-2*t) / torch.sqrt(2*t*(1-t))) + v_tornado 

    opt_pot.zero_grad()
    v_pred_pot = get_gradient_field(net_potential, x_t, t)
    loss_pot = torch.mean((v_pred_pot - v_target)**2)
    loss_pot.backward()
    opt_pot.step()
    
    opt_dir.zero_grad()
    v_pred_dir = net_direct(x_t, t)
    loss_dir = torch.mean((v_pred_dir - v_target)**2)
    loss_dir.backward()
    opt_dir.step()
    
    if (epoch+1) % 500 == 0:
        print(f"Epoch {epoch+1:4d} | Scalar loss: {loss_pot.item():.4f} | Direct loss: {loss_dir.item():.4f}")

In [ ]:
def velocity_pot(x, t):
    with torch.enable_grad():
        x_in = x.detach().requires_grad_(True)
        v = get_gradient_field(net_potential, x_in, t)
    return v.detach()

@torch.no_grad()
def velocity_dir(x, t):
    return net_direct(x, t)

In [ ]:
print("Running inference...")

n_test = 2500
x_start = torch.randn(n_test, 2)
steps = 15

x_gen_pot = solve_ode(x_start, velocity_pot, steps=steps)
x_gen_dir = solve_ode(x_start, velocity_dir, steps=steps)

print("Inference done!")

In [ ]:
x_true_np, _ = make_moons(n_samples=batch_size, noise=0.05, random_state=42)
x_true = torch.tensor((x1_np - np.array([0.5, 0.25])) * 2.0, dtype=torch.float32)

x_start_np = x_start.cpu().numpy()
x_true_np = x_true.cpu().numpy()
x_pot_np = x_gen_pot.cpu().numpy()
x_dir_np = x_gen_dir.cpu().numpy()

def plot_scatter(ax, data, title):
    ax.scatter(data[:, 0], data[:, 1], s=4, alpha=0.3, color='black')
    ax.set_title(title, fontsize=14)
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_aspect('equal')
    ax.grid(True, linestyle=':', alpha=0.5)

In [ ]:
fig1, axes1 = plt.subplots(1, 2, figsize=(10, 5))

plot_scatter(axes1[0], x_start_np, "Initial Data X_0")
plot_scatter(axes1[1], x_true_np, "Target X_1")

fig1.tight_layout()
plt.savefig('X_0_vs_X_1', dpi=600)
plt.show()

In [ ]:
fig2, axes2 = plt.subplots(1, 2, figsize=(10, 5))

plot_scatter(axes2[0], x_pot_np, "Generated (PotentialNet)")
plot_scatter(axes2[1], x_dir_np, "Generated (DirectNet)")

fig2.tight_layout()
fig2.savefig('experiment3.png', dpi=600)
plt.show()